# Consolidación SIPSA-DANE – Precios mayoristas (2013 – 2026)

Este notebook consolida todos los archivos Excel de la carpeta
`raw_data/Bases de Datos SIPSA-DANE Completo` en un único archivo CSV:

`processed_data/sipsa_precios_mayoristas_consolidado.csv`

**Consideraciones de los archivos fuente:**
- `series-historicas-precios-mayoristas 2013-2017.xlsx` trae un año por hoja (`1.1` = 2013 … `1.5` = 2017).
- 2019 – 2023 traen una hoja por mes; 2018 y 2024 – 2026 traen una sola hoja con todo el año.
- La fila de encabezado cambia de posición según el archivo, así que se detecta automáticamente (la fila cuya primera celda es `Fecha`).
- Los nombres de columnas cambian entre años (`Fuente` / `Mercado`, `Precio` / `Precio promedio por kilogramo*`), por lo que se estandarizan.
- Las hojas auxiliares (`Índice`, `CPC`, `Fuentes`) se omiten.

Se puede ejecutar en **Google Colab** (clona el repositorio público) o de forma local desde la carpeta `Data` del repositorio.

## 1. Preparar el entorno

In [ ]:
import os
import sys

REPO_URL = "https://github.com/yevasa93/AnomaliasClimaticasArrozCarne.git"
REPO_DIR = "AnomaliasClimaticasArrozCarne"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # En Colab se clona el repositorio público (o se actualiza si ya existe)
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !git -C {REPO_DIR} pull
    os.chdir(REPO_DIR)

print("Directorio de trabajo:", os.getcwd())

In [ ]:
import glob
import unicodedata

import pandas as pd

RAW_DIR = os.path.join("raw_data", "Bases de Datos SIPSA-DANE Completo")
OUT_DIR = "processed_data"
OUT_FILE = os.path.join(OUT_DIR, "sipsa_precios_mayoristas_consolidado.csv")

os.makedirs(OUT_DIR, exist_ok=True)

archivos = sorted(glob.glob(os.path.join(RAW_DIR, "*.xlsx")))
print(f"{len(archivos)} archivos encontrados:")
for a in archivos:
    print("  -", os.path.basename(a))

## 2. Funciones de lectura y estandarización

Cada hoja se lee sin encabezado, se ubica la fila donde la primera celda es `Fecha` y se toma como encabezado.
Luego se renombran las columnas a un esquema común:

| Columna final | Nombres originales |
|---|---|
| `Fecha` | Fecha |
| `Grupo` | Grupo |
| `Producto` | Producto |
| `Mercado` | Fuente / Mercado |
| `Precio_kg` | Precio / Precio por kilogramo* / Precio promedio por kilogramo* |

Las columnas que solo existen en los archivos más recientes (Código CPC, Departamento, Municipio, etc.) no se incluyen porque no están disponibles para toda la serie.

In [ ]:
COLUMNAS_FINALES = ["Fecha", "Grupo", "Producto", "Mercado", "Precio_kg"]


def normalizar_texto(texto):
    """Minúsculas, sin tildes y sin espacios repetidos (para comparar nombres de columnas)."""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.lower().split())


def mapear_columna(nombre):
    n = normalizar_texto(nombre)
    if n == "fecha":
        return "Fecha"
    if n == "grupo":
        return "Grupo"
    if n == "producto":
        return "Producto"
    if n in ("fuente", "mercado"):
        return "Mercado"
    if n.startswith("precio"):
        return "Precio_kg"
    return None  # columna que no se conserva


def leer_hoja(df_crudo, archivo, hoja):
    """Recibe una hoja leída sin encabezado y devuelve un DataFrame estandarizado (o None)."""
    primera_col = df_crudo.iloc[:30, 0].astype(str).str.strip()
    filas_encabezado = primera_col.index[primera_col == "Fecha"]
    if len(filas_encabezado) == 0:
        return None  # hoja auxiliar (Índice, CPC, Fuentes...)

    fila = filas_encabezado[0]
    df = df_crudo.iloc[fila + 1:].copy()
    df.columns = [mapear_columna(c) for c in df_crudo.iloc[fila]]
    df = df.loc[:, [c is not None for c in df.columns]]

    faltantes = set(COLUMNAS_FINALES) - set(df.columns)
    if faltantes:
        raise ValueError(f"{archivo} / {hoja}: faltan columnas {faltantes}")

    df = df[COLUMNAS_FINALES]
    df["Archivo_origen"] = archivo
    df["Hoja_origen"] = str(hoja)
    return df

## 3. Lectura de todos los archivos y hojas

In [ ]:
partes = []
resumen = []

for ruta in archivos:
    archivo = os.path.basename(ruta)
    hojas = pd.read_excel(ruta, sheet_name=None, header=None)  # todas las hojas
    for hoja, df_crudo in hojas.items():
        df = leer_hoja(df_crudo, archivo, hoja)
        if df is None:
            resumen.append((archivo, hoja, "omitida", 0))
            continue
        partes.append(df)
        resumen.append((archivo, hoja, "leída", len(df)))

resumen = pd.DataFrame(resumen, columns=["Archivo", "Hoja", "Estado", "Filas_crudas"])
datos = pd.concat(partes, ignore_index=True)

print(f"Hojas leídas: {(resumen.Estado == 'leída').sum()} | omitidas: {(resumen.Estado == 'omitida').sum()}")
print(f"Filas crudas consolidadas: {len(datos):,}")
resumen

## 4. Limpieza

- Conversión de `Fecha` a fecha y `Precio_kg` a número.
- Eliminación de filas vacías o notas al pie (sin fecha válida o sin precio).
- Limpieza de espacios en los textos.
- Eliminación de registros duplicados.

In [ ]:
df = datos.copy()

df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
df["Precio_kg"] = pd.to_numeric(df["Precio_kg"], errors="coerce")

for col in ["Grupo", "Producto", "Mercado"]:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

filas_antes = len(df)
df = df.dropna(subset=["Fecha", "Producto", "Mercado", "Precio_kg"])
print(f"Filas eliminadas por fecha/precio/producto/mercado vacío (notas al pie, filas en blanco): {filas_antes - len(df):,}")

# El grupo se unifica en mayúsculas y sin tildes
# (en algunos archivos aparece "TUBÉRCULOS, RAÍCES Y PLÁTANOS" y en otros "TUBERCULOS, RAICES Y PLATANOS")
df["Grupo"] = df["Grupo"].map(lambda g: normalizar_texto(g).upper()).astype("string")

filas_antes = len(df)
df = df.drop_duplicates(subset=["Fecha", "Grupo", "Producto", "Mercado", "Precio_kg"])
print(f"Duplicados eliminados: {filas_antes - len(df):,}")

df["Anio"] = df["Fecha"].dt.year
df["Mes"] = df["Fecha"].dt.month

df = df[["Fecha", "Anio", "Mes", "Grupo", "Producto", "Mercado", "Precio_kg", "Archivo_origen", "Hoja_origen"]]
df = df.sort_values(["Fecha", "Grupo", "Producto", "Mercado"]).reset_index(drop=True)

print(f"Filas finales: {len(df):,}")
df.head()

## 5. Validaciones

In [ ]:
print("Rango de fechas:", df["Fecha"].min().date(), "a", df["Fecha"].max().date())

validacion = df.groupby("Anio").agg(
    registros=("Precio_kg", "size"),
    meses=("Mes", "nunique"),
    productos=("Producto", "nunique"),
    mercados=("Mercado", "nunique"),
)
validacion

In [ ]:
# Registros por grupo de alimentos
df["Grupo"].value_counts()

In [ ]:
# Revisión de productos de interés para el proyecto (arroz y carnes)
interes = df[df["Producto"].str.contains("arroz", case=False) | (df["Grupo"] == "CARNES")]
interes.groupby(["Grupo", "Producto"]).size().sort_values(ascending=False).head(30)

> **Nota sobre 2026:** a partir de 2026 SIPSA incluye además precios en **molinos de arroz** (p. ej. *Espinal, Fedearroz*, *Ibagué, Molino Federal*), por eso ese año tiene más mercados que los anteriores.

## 6. Exportar CSV

Las columnas `Archivo_origen` y `Hoja_origen` sirven para la trazabilidad dentro del notebook, pero no se exportan:
agregan cerca de 30 MB al archivo y GitHub no admite archivos de más de 100 MB.

In [ ]:
COLUMNAS_CSV = ["Fecha", "Anio", "Mes", "Grupo", "Producto", "Mercado", "Precio_kg"]
df[COLUMNAS_CSV].to_csv(OUT_FILE, index=False, encoding="utf-8", date_format="%Y-%m-%d")

tamano_mb = os.path.getsize(OUT_FILE) / 1024 ** 2
print(f"Archivo generado: {OUT_FILE} ({tamano_mb:.1f} MB, {len(df):,} filas)")
if tamano_mb > 100:
    print("ADVERTENCIA: GitHub no acepta archivos mayores a 100 MB.")

In [ ]:
# En Colab: descargar el CSV generado para subirlo al repositorio
if EN_COLAB:
    from google.colab import files
    files.download(OUT_FILE)

## 7. Lectura del CSV desde el repositorio público de GitHub

Una vez el CSV está subido al repositorio, se puede leer desde cualquier notebook así:

In [ ]:
URL_CSV = (
    "https://raw.githubusercontent.com/yevasa93/AnomaliasClimaticasArrozCarne/main/"
    "processed_data/sipsa_precios_mayoristas_consolidado.csv"
)

sipsa = pd.read_csv(URL_CSV, parse_dates=["Fecha"])
print(sipsa.shape)
sipsa.head()